# AI Agent — LangChain Pandas Agent

Uses a LangChain agent backed by an LLM to answer natural language questions about the dataset.

## Setup

Create a `.env` file in the project root (never commit this file):
```
ANTHROPIC_API_KEY=your_key_here
GROQ_API_KEY=your_key_here
```

**Dataset:** [Online Retail II](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) — place the file at `data/online_retail_II.xlsx`

In [1]:
import subprocess
subprocess.run(["pip", "install", "langchain", "langchain-anthropic", "langchain-groq", "langchain-experimental", "python-dotenv"], check=True)

You should consider upgrading via the '/Users/houdabensidiahmed/Documents/ML project/venv/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['pip', 'install', 'langchain', 'langchain-anthropic', 'langchain-groq', 'langchain-experimental', 'python-dotenv'], returncode=0)

## 1. Load API Keys

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads from .env file

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not ANTHROPIC_API_KEY and not GROQ_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY or GROQ_API_KEY in a .env file")

## 2. Initialise LLM

Uncomment the provider you want to use.

In [3]:
# Option A — Anthropic Claude (uncomment to use)
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-haiku-20240307", api_key=ANTHROPIC_API_KEY)

# Option B — Groq (faster, free tier available)
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

/Users/houdabensidiahmed/Documents/ML project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


LangChain is an open-source framework that allows developers to build applications on top of large language models, providing a set of tools and APIs to interact with these models and create custom interfaces and workflows.


## 3. Create Pandas DataFrame Agent

In [4]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent

df = pd.read_excel("data/online_retail_II.xlsx")

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=15,
    max_execution_time=30,
    handle_parsing_errors=True  # handles cases where LLM output format is unexpected
)

/Users/houdabensidiahmed/Documents/ML project/venv/lib/python3.9/site-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


## 4. Ask Questions

In [5]:
response = agent.invoke("Which country had the highest revenue?")
print(response['output'])



> Entering new AgentExecutor chain...


Thought: To find the country with the highest revenue, we need to first calculate the total revenue for each country. This can be done by multiplying the Quantity and Price columns for each row, and then grouping the results by Country.

Action: python_repl_ast
Action Input: 
```python
df['Revenue'] = df['Quantity'] * df['Price']
country_revenue = df.groupby('Country')['Revenue'].sum()
print(country_revenue)
```Country
Australia                 30051.800
Austria                   13023.280
Bahrain                    2313.150
Belgium                   22663.530
Bermuda                    1253.140
Brazil                      268.270
Canada                     1216.660
Channel Islands           21367.320
Cyprus                    11216.450
Denmark                   46972.950
EIRE                     352242.730
Finland                    7187.710
France                   130769.900
Germany                  196290.351
Greece                    14284.970
Hong Kong                  3922.260
I

Thought: Now that we have the total revenue for each country, we can find the country with the highest revenue by finding the maximum value in the country_revenue series.

Action: python_repl_ast
Action Input: 
```python
max_revenue_country = country_revenue.idxmax()
print(max_revenue_country)
```United Kingdom


I now know the final answer

Final Answer: The country with the highest revenue is the United Kingdom.

> Finished chain.
The country with the highest revenue is the United Kingdom.


In [6]:
response = agent.invoke("What are the top 5 best selling products by revenue? Print the product names and their total revenue.")
print(response['output'])



> Entering new AgentExecutor chain...


Thought: To find the top 5 best-selling products by revenue, we need to calculate the total revenue for each product and then sort them in descending order. The revenue for each product can be calculated by multiplying the quantity sold by the price of the product.

Action: python_repl_ast
Action Input:
```python
import pandas as pd

# Calculate revenue for each row
df['Revenue'] = df['Quantity'] * df['Price']

# Group by 'Description' and calculate total revenue for each product
product_revenue = df.groupby('Description')['Revenue'].sum().reset_index()

# Sort products by total revenue in descending order and get top 5
top_5_products = product_revenue.sort_values('Revenue', ascending=False).head(5)

print(top_5_products)
```                             Description    Revenue
3371            REGENCY CAKESTAND 3 TIER  163051.46
4402  WHITE HANGING HEART T-LIGHT HOLDER  157865.43
1297                      DOTCOM POSTAGE  116401.99
279        ASSORTED COLOUR BIRD ORNAMENT   72454.12
2780 

The observation shows the top 5 best-selling products by revenue, along with their total revenue. 

Action: python_repl_ast
Action Input:
```python
print(top_5_products)
```                             Description    Revenue
3371            REGENCY CAKESTAND 3 TIER  163051.46
4402  WHITE HANGING HEART T-LIGHT HOLDER  157865.43
1297                      DOTCOM POSTAGE  116401.99
279        ASSORTED COLOUR BIRD ORNAMENT   72454.12
2780     PAPER CHAIN KIT 50'S CHRISTMAS    57870.20


I now know the final answer

The top 5 best-selling products by revenue are:
1. REGENCY CAKESTAND 3 TIER - $163,051.46
2. WHITE HANGING HEART T-LIGHT HOLDER - $157,865.43
3. DOTCOM POSTAGE - $116,401.99
4. ASSORTED COLOUR BIRD ORNAMENT - $72,454.12
5. PAPER CHAIN KIT 50'S CHRISTMAS - $57,870.20

Final Answer: 
The top 5 best-selling products by revenue are:
1. REGENCY CAKESTAND 3 TIER - $163,051.46
2. WHITE HANGING HEART T-LIGHT HOLDER - $157,865.43
3. DOTCOM POSTAGE - $116,401.99
4. ASSORTED COLOUR BIRD ORNAMENT - $72,454.12
5. PAPER CHAIN KIT 50'S CHRISTMAS - $57,870.20

> Finished chain.
The top 5 best-selling products by revenue are:
1. REGENCY CAKESTAND 3 TIER - $163,051.46
2. WHITE HANGING HEART T-LIGHT HOLDER - $157,865.43
3. DOTCOM POSTAGE - $116,401.99
4. ASSORTED COLOUR BIRD ORNAMENT - $72,454.12
5. PAPER CHAIN KIT 50'S CHRISTMAS - $57,870.20
